# 11.1 DINOv3 spatial patch pooling

Frozen representation ablation: CLS token versus CLS concatenated with mean spatial patch features.

Method references: [DINOv3 paper](https://arxiv.org/abs/2508.10104), [official implementation](https://github.com/facebookresearch/dinov3), and [Transformers DINOv3 interface](https://huggingface.co/docs/transformers/en/model_doc/dinov3). Patch mean pooling is our declared readout, not a reproduced dental method or a saliency explanation.

```mermaid
flowchart LR
A[Locked raw ROI data] --> B[Official DINOv3 encoder]
B --> C[Declared representation / adaptation]
C --> D[Train-only fitting and validation selection]
D --> E[Per-class metrics and saved evidence]
```

## 0. Load the experiment tools

These imports support image inspection, feature extraction, train-only scaling, validation metrics, plots, and output checks. The notebook runs on Kaggle T4 x2; the local machine only prepares the source.

In [ ]:
from pathlib import Path
import os,sys,subprocess,hashlib,json,time,platform
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display,Markdown
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,precision_recall_fscore_support
import joblib,torch
def save_json(path,value):Path(path).write_text(json.dumps(value,indent=2),encoding='utf-8')

## 1. Define a reproducible, smoke-gated run

The same frozen encoder produces two readouts. `CLS` summarizes the crop; `CLS + mean patch` appends the average of spatial patch tokens. No encoder weights are updated. Kaggle runs use the predeclared Hugging Face revision and raw ROI dataset version 1. Test data is excluded.

In [ ]:
RUN_EXPERIMENT = False  # The Kaggle staging script enables this only for execution.
PHASE = 'smoke'         # Staging selects smoke first, then full after review.
SEED = 42
MODEL_ID = 'facebook/dinov3-vits16-pretrain-lvd1689m'
MODEL_REVISION = '114c1379950215c8b35dfcd4e90a5c251dde0d32'
EXPECTED_MANIFEST_SHA256 = 'b9feaaceb4b9ea94ad540b553927ffb7f67e731098f7a78c770941804cbb4f06'
DATASET_VERSION = 'trihuynhviprovcl/dentalselfie-gum-raw-roi-v1/1'
OUTPUT = Path('/kaggle/working/dentalselfie-11-1')
TEMP = Path('/kaggle/temp/dentalselfie-11-1')
CONFIG = {'image_size':224, 'patch_size':16, 'batch_size':64,
          'c_grid':[0.01,0.1,1.0,10.0], 'seed':SEED, 'phase':PHASE,
          'model_id':MODEL_ID, 'model_revision':MODEL_REVISION,
          'manifest_sha256':EXPECTED_MANIFEST_SHA256}
display(pd.Series(CONFIG).to_frame('fixed setting'))
print('Run enabled:', RUN_EXPERIMENT)

## 2. Install and record the DINOv3 interface

Use the Transformers implementation already used for the successful probe. The four pinned Hugging Face packages are installed without dependencies into a session-local directory. Existing Kaggle Torch, NumPy and scientific libraries are reused. This avoids changing the global Hub version required by unrelated Gradio/Diffusers packages. Display and assert the actual import paths; no access token is involved in installation.

In [ ]:
# Install only the HF stack into a session-local directory, not global site-packages.
RUNTIME_DIR=Path('/kaggle/temp/dentalselfie_hf_runtime')
RUNTIME_PACKAGES=['transformers==4.57.1','huggingface-hub==0.36.2',
                  'tokenizers==0.22.1','safetensors==0.6.2']
if RUN_EXPERIMENT:
    import importlib
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-deps','--upgrade',
                    '--target',str(RUNTIME_DIR),*RUNTIME_PACKAGES],check=True)
    sys.path.insert(0,str(RUNTIME_DIR))
    os.environ['PYTHONPATH']=str(RUNTIME_DIR)+os.pathsep+os.environ.get('PYTHONPATH','')
    importlib.invalidate_caches()
    import transformers,huggingface_hub,tokenizers,safetensors
    for module in [transformers,huggingface_hub,tokenizers,safetensors]:
        assert Path(module.__file__).resolve().is_relative_to(RUNTIME_DIR.resolve()), module.__name__
    assert transformers.__version__=='4.57.1' and huggingface_hub.__version__=='0.36.2'
    CONFIG['runtime_packages']=RUNTIME_PACKAGES
    display(pd.DataFrame([{'package':m.__name__,'version':m.__version__,'path':m.__file__}
                          for m in [transformers,huggingface_hub,tokenizers,safetensors]]))
else:
    print('HF runtime installation is disabled in the source preview.')

import torch, torchvision
print({'python':platform.python_version(),'torch':torch.__version__,
       'torchvision':torchvision.__version__,'cuda':torch.version.cuda,
       'gpu_count':torch.cuda.device_count(),
       'gpu_names':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})

## 3. Find and verify the immutable input manifest

The raw-source manifest locks the 10,889 crop identities and train/validation assignments. Hash the manifest and every requested image before using it. Do not discover a test directory or include test rows.

In [ ]:
def sha256_file(path):
    digest=hashlib.sha256()
    with open(path,'rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):digest.update(block)
    return digest.hexdigest()

manifests=list(Path('/kaggle/input').rglob('manifest.csv')) if Path('/kaggle/input').exists() else []
matched=[path for path in manifests if sha256_file(path)==EXPECTED_MANIFEST_SHA256]
if RUN_EXPERIMENT and len(matched)!=1:
    raise RuntimeError(f'Expected exactly one pinned raw ROI manifest, found {len(matched)}')
if matched:
    DATA_ROOT=matched[0].parent
    frame=pd.read_csv(matched[0])
    assert set(frame.split)=={'train','val'} and frame.crop_id.is_unique
    assert frame.groupby('group_id').split.nunique().max()==1
    assert (frame.groupby(['split','class_id']).size().unstack(fill_value=0).index==['train','val']).all()
    assert frame.class_id.isin([0,1,2]).all()
    print('Manifest:',len(frame),'rows | hash:',sha256_file(matched[0]),'| test rows: 0')
    display(frame.groupby(['split','class_name']).agg(crops=('crop_id','size'),groups=('group_id','nunique')))
else:
    DATA_ROOT=None
    print('Local preview only. Attach the pinned raw ROI input in Kaggle to execute.')

## 4. Inspect the exact 224-pixel crops

This panel shows the model input as stored. The same deterministic resize and ImageNet normalization are applied to both representations. No additional center crop or context image is introduced.

In [ ]:
if DATA_ROOT is not None:
    examples=frame.groupby('class_id',group_keys=False).head(2)
    fig,axes=plt.subplots(3,2,figsize=(7,9))
    for ax,row in zip(axes.flat,examples.itertuples()):
        with Image.open(DATA_ROOT/row.local_path) as image:ax.imshow(image.convert('RGB'))
        ax.set_title(f'{row.class_name} | {row.crop_id}',fontsize=8);ax.axis('off')
    fig.tight_layout();plt.show()
else:
    display(Markdown('Preview appears after the Kaggle input is attached.'))

## 5. Load official weights without exposing the secret

The repository is gated. Retrieve `HF_TOKEN` through Kaggle Secrets; never print, save, or embed it in a notebook. If the secret is not attached or access is not granted, stop with an actionable error. Record only the immutable model revision and model configuration.

In [ ]:
def get_hf_token():
    token=os.environ.get('HF_TOKEN')
    if token:return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('Attach the HF_TOKEN secret to this Kaggle notebook and accept official DINOv3 access.') from exc

encoder=None
if RUN_EXPERIMENT:
    from transformers import AutoModel
    token=get_hf_token()
    encoder=AutoModel.from_pretrained(MODEL_ID,revision=MODEL_REVISION,token=token,
                                      use_safetensors=True,trust_remote_code=False).eval().to('cuda:0')
    del token
    assert encoder.config.patch_size==16 and encoder.config.hidden_size==384
    print({'resolved_revision':encoder.config._commit_hash,'hidden_size':encoder.config.hidden_size,
           'patch_size':encoder.config.patch_size,'register_tokens':encoder.config.num_register_tokens,
           'transformer_blocks':encoder.config.num_hidden_layers})
    assert encoder.config._commit_hash==MODEL_REVISION

## 6. Recreate the image preprocessing and patch pooling

DINOv3 outputs one CLS token, register tokens, then spatial patch tokens. Exclude the register tokens before averaging patches. At 224 pixels and patch size 16, the expected grid is 14 x 14. The second readout concatenates the 384-dimensional CLS and 384-dimensional mean patch vector.

In [ ]:
transform=transforms.Compose([
    transforms.Resize((224,224),interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
def pool_dinov3_tokens(outputs,model,pixel_values):
    tokens=outputs.last_hidden_state
    batch,_,height,width=pixel_values.shape
    patch_size=model.config.patch_size
    patch_h,patch_w=height//patch_size,width//patch_size
    prefix=1+model.config.num_register_tokens
    expected=prefix+patch_h*patch_w
    assert tokens.shape==(batch,expected,model.config.hidden_size),tokens.shape
    cls=tokens[:,0,:]
    patches=tokens[:,prefix:,:]
    assert patches.shape[1]==patch_h*patch_w
    patch_grid=patches.unflatten(1,(patch_h,patch_w))
    patch_mean=patch_grid.mean(dim=(1,2))
    combined=torch.cat([cls,patch_mean],dim=1)
    return cls,patch_mean,combined,patch_grid
print('224 / 16 = 14 x 14 spatial patches; register tokens excluded from pooling.')

## 7. Extract deterministic features from train and validation only

Full runs verify each crop checksum, keep source groups inside their locked split, and cache feature arrays under `/kaggle/temp`. Smoke uses a balanced, deterministic sample to check every class and code path; smoke metrics are diagnostic only.

In [ ]:
def selected_rows(split):
    rows=frame[frame.split.eq(split)].copy()
    if PHASE=='smoke':
        rows=rows.groupby('class_id',group_keys=False).head(16).copy()
    return rows.reset_index(drop=True)

class CropDataset(Dataset):
    def __init__(self,rows,verify):self.rows=rows;self.verify=verify
    def __len__(self):return len(self.rows)
    def __getitem__(self,index):
        row=self.rows.iloc[index];path=DATA_ROOT/row.local_path
        if self.verify and sha256_file(path)!=row.local_sha256:raise ValueError(f'Checksum mismatch: {row.crop_id}')
        with Image.open(path) as image:x=transform(image.convert('RGB'))
        return x,int(row.class_id),str(row.crop_id)

def extract_split(split):
    rows=selected_rows(split)
    loader=DataLoader(CropDataset(rows,verify=(PHASE=='full')),batch_size=CONFIG['batch_size'],
                      shuffle=False,num_workers=2,pin_memory=True)
    outputs=[];labels=[];ids=[];patch_example=None
    started=time.monotonic()
    with torch.inference_mode():
        for step,(images,y,crop_ids) in enumerate(loader):
            images=images.cuda(non_blocking=True)
            with torch.autocast('cuda',dtype=torch.float16):encoded=encoder(pixel_values=images)
            cls,patch_mean,combined,patch_grid=pool_dinov3_tokens(encoded,encoder,images)
            outputs.append(torch.cat([cls.float(),patch_mean.float(),combined.float()],dim=1).cpu().numpy())
            labels.extend(y.tolist());ids.extend(crop_ids)
            if patch_example is None:patch_example=(images[0].cpu(),patch_grid[0].float().norm(dim=-1).cpu())
            if step%20==0:print(f'{split}: {step+1}/{len(loader)} batches',flush=True)
    features=np.concatenate(outputs)
    assert len(features)==len(rows) and set(labels)=={0,1,2}
    print(split,'rows',len(rows),'feature extraction seconds',round(time.monotonic()-started,1))
    return rows,features,np.asarray(labels),ids,patch_example

if RUN_EXPERIMENT:
    TEMP.mkdir(parents=True,exist_ok=True)
    extracted={split:extract_split(split) for split in ['train','val']}
    train_rows,train_all,y_train,id_train,_=extracted['train']
    val_rows,val_all,y_val,id_val,patch_example=extracted['val']
    print('Output blocks: CLS 384 + patch mean 384; concatenated 768.')

## 8. Visualize patch features as a representation diagnostic

The patch norm map shows feature magnitude only. It is not an attention map, saliency map, segmentation, or clinical explanation. It is included to make the 14 x 14 spatial feature grid tangible.

In [ ]:
if RUN_EXPERIMENT:
    image_tensor,feature_map=patch_example
    image=(image_tensor*torch.tensor([.229,.224,.225])[:,None,None]+
           torch.tensor([.485,.456,.406])[:,None,None]).clamp(0,1).permute(1,2,0).numpy()
    fig,axes=plt.subplots(1,2,figsize=(9,4))
    axes[0].imshow(image);axes[0].set_title('224 x 224 crop');axes[0].axis('off')
    axes[1].imshow(image);axes[1].imshow(feature_map.numpy(),alpha=.55,cmap='magma',
        extent=(0,224,224,0),interpolation='nearest');axes[1].set_title('Patch feature norm (not saliency)');axes[1].axis('off')
    fig.tight_layout();plt.show()

## 9. Fit matched linear probes and select regularization on validation

Both readouts use the same train-fitted StandardScaler, LogisticRegression implementation, four-value C grid, seed, rows, and validation selection metric. Validation is repeatedly reused elsewhere in the project, so these results are exploratory. Test data stays unopened.

In [ ]:
def metric_table(y,pred):
    cm=confusion_matrix(y,pred,labels=[0,1,2])
    precision,recall,f1,support=precision_recall_fscore_support(y,pred,labels=[0,1,2],zero_division=0)
    return {'accuracy':float(accuracy_score(y,pred)),'macro_f1':float(f1.mean()),
            'balanced_accuracy':float(recall.mean()),'class_precision':precision.tolist(),
            'class_recall':recall.tolist(),'class_f1':f1.tolist(),'support':support.tolist(),
            'confusion_matrix':cm.tolist()}

if RUN_EXPERIMENT:
    OUTPUT.mkdir(parents=True,exist_ok=True)
    feature_specs={'CLS':(0,384),'CLS_plus_patch_mean':(768,1536)}
    results=[];predictions={};grid_rows=[]
    for name,(start,end) in feature_specs.items():
        train_features=train_all[:,start:end];val_features=val_all[:,start:end]
        scaler=StandardScaler().fit(train_features)
        train_x=scaler.transform(train_features);val_x=scaler.transform(val_features)
        grid=[];best=None
        for c in CONFIG['c_grid']:
            classifier=LogisticRegression(C=c,max_iter=1000,solver='lbfgs',random_state=SEED)
            classifier.fit(train_x,y_train);pred=classifier.predict(val_x)
            metrics=metric_table(y_val,pred);grid.append({'feature':name,'C':c,**metrics})
            grid_rows.append({'feature':name,'C':c,**metrics})
            if best is None or metrics['macro_f1']>best['metrics']['macro_f1']:
                best={'classifier':classifier,'scaler':scaler,'C':c,'metrics':metrics,'pred':pred}
        best['metrics'].update(feature=name,selected_C=best['C'],feature_dim=end-start,
            model_id=MODEL_ID,model_revision=MODEL_REVISION,manifest_sha256=EXPECTED_MANIFEST_SHA256,
            seed=SEED,phase=PHASE,test_images_opened=0)
        results.append(best['metrics']);predictions[name]=best['pred']
        pd.DataFrame({'crop_id':id_val,'true_class':y_val,'predicted_class':best['pred'],
          'probabilities':[json.dumps(p.tolist()) for p in best['classifier'].predict_proba(val_x)]}).to_csv(
          OUTPUT/f'{name}_validation_predictions.csv',index=False)
        save_json(OUTPUT/f'{name}_metrics.json',best['metrics'])
        joblib.dump({'scaler':best['scaler'],'classifier':best['classifier']},OUTPUT/f'{name}_probe.joblib')
    grid_frame=pd.DataFrame(grid_rows);grid_frame.to_csv(OUTPUT/'probe_grid.csv',index=False)
    result_frame=pd.DataFrame([{k:v for k,v in row.items() if not isinstance(v,list)} for row in results])
    result_frame.to_csv(OUTPUT/'comparison.csv',index=False)
    display(result_frame.style.format({'accuracy':'{:.2%}','macro_f1':'{:.2%}','balanced_accuracy':'{:.2%}'}))
    fig,ax=plt.subplots(figsize=(8,4))
    for name,group in grid_frame.groupby('feature'):
        ax.plot(group.C,group.macro_f1,marker='o',label=name)
    ax.set_xscale('log');ax.set(xlabel='C (log scale)',ylabel='Validation macro-F1',ylim=(0,1),title='Matched frozen-feature probe');ax.legend();fig.tight_layout();plt.show()

## 10. Compare per-class behavior and save the experiment record

Read the confusion matrices and recall of each class alongside accuracy and macro-F1. A higher aggregate score is not enough if `no-visible` behavior collapses. The probe grid, predictions, head, configuration, and resolved DINO revision are saved for collection.

In [ ]:
if RUN_EXPERIMENT:
    for result in results:
        display(Markdown('**'+result['feature']+' | validation confusion matrix**'))
        cm=np.asarray(result['confusion_matrix'])
        fig,ax=plt.subplots(figsize=(4,3));display_matrix=ax.imshow(cm,cmap='Blues')
        ax.set(xticks=[0,1,2],yticks=[0,1,2],xlabel='Predicted',ylabel='Reference',title=result['feature'])
        for (i,j),value in np.ndenumerate(cm):ax.text(j,i,int(value),ha='center',va='center')
        fig.colorbar(display_matrix,ax=ax);fig.tight_layout();plt.show()
        display(pd.DataFrame({'class':['no-visible','mild','marked'],
                              'precision':result['class_precision'],'recall':result['class_recall'],
                              'F1':result['class_f1']}))
    save_json(OUTPUT/'run_config.json',CONFIG)
    print('Saved:',sorted(p.name for p in OUTPUT.iterdir()))
else:
    display(Markdown('Execution is disabled in the published source; the Kaggle runner enables a smoke pass before the full feature extraction.'))